# 01 — Data Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config
from src.data_loading import load_ratings, load_items, describe_dataset, train_test_split_ratings

## 1. Load the data

In [ ]:
ratings = load_ratings()
items = load_items()

print(f"Ratings shape: {ratings.shape}")
print(f"Items shape:   {items.shape}")

In [ ]:
ratings.head()

In [ ]:
items.head()

## 2. Dataset statistics

We compute the core numbers that characterise any recommender dataset: number of users, items, interactions, and sparsity.

**Sparsity** is the fraction of possible user-item pairs that have no rating. In practice, sparsity is always very high — most users have only rated a small subset of the catalog. This is one of the main challenges recommender systems need to address.

In [ ]:
describe_dataset(ratings, items)

In [ ]:
n_users = ratings[config.USER_COL].nunique()
n_items = ratings[config.ITEM_COL].nunique()
n_ratings = len(ratings)
sparsity = 1 - n_ratings / (n_users * n_items)

print(f"Users    : {n_users:,}")
print(f"Movies   : {n_items:,}")
print(f"Ratings  : {n_ratings:,}")
print(f"Sparsity : {sparsity:.2%}")

The dataset has **610 users**, **9,724 movies**, and **100,836 ratings**. The sparsity is ~98.3%, meaning that out of all possible user-movie combinations, only 1.7% actually have a rating. This is typical for movie datasets and is precisely the problem recommender systems are designed to solve.

## 3. Rating distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

dist = ratings[config.RATING_COL].value_counts().sort_index()
ax.bar(dist.index.astype(str), dist.values, color='steelblue', edgecolor='white')
ax.set_xlabel('Rating')
ax.set_ylabel('Number of ratings')
ax.set_title('Rating distribution — MovieLens Small')
plt.tight_layout()
plt.savefig('results/figures/rating_distribution.png', dpi=150)
plt.show()

print(f"Mean rating : {ratings[config.RATING_COL].mean():.2f}")
print(f"Median      : {ratings[config.RATING_COL].median():.1f}")

The distribution is left-skewed: most ratings are between 3 and 4.5. The mean rating is 3.50. This is typical: users tend to rate movies they chose to watch, so there is a natural positive bias (they were somewhat interested already).

## 4. Most active users

In [ ]:
user_counts = ratings.groupby(config.USER_COL).size().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(user_counts.values, bins=40, color='steelblue', edgecolor='white')
ax.set_xlabel('Number of ratings per user')
ax.set_ylabel('Number of users')
ax.set_title('Ratings per user distribution')
plt.tight_layout()
plt.savefig('results/figures/ratings_per_user.png', dpi=150)
plt.show()

print("Top 5 most active users:")
print(user_counts.head())

The distribution of ratings per user is highly skewed. A small number of users have rated thousands of movies, while the majority have rated fewer than 100. This long-tail distribution is common and has implications for collaborative filtering: users with very few ratings are harder to model.

## 5. Most popular movies

In [ ]:
item_counts = (
    ratings.groupby(config.ITEM_COL)
    .size()
    .reset_index(name='n_ratings')
    .merge(items[[config.ITEM_COL, config.TITLE_COL]], on=config.ITEM_COL)
    .sort_values('n_ratings', ascending=False)
)

top10 = item_counts.head(10)

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(top10[config.TITLE_COL].str[:40], top10['n_ratings'], color='steelblue')
ax.invert_yaxis()
ax.set_xlabel('Number of ratings')
ax.set_title('Top 10 most rated movies')
plt.tight_layout()
plt.savefig('results/figures/most_rated_movies.png', dpi=150)
plt.show()

The catalog also follows a long-tail pattern: a few blockbusters like Forrest Gump and The Shawshank Redemption have hundreds of ratings, while the vast majority of movies have fewer than 10. This popularity bias is something we need to keep in mind when evaluating models: a recommender that just recommends popular items will score well on some metrics but offers little personalisation.

## 6. Genre distribution

In [ ]:
genre_series = items[config.GENRES_COL].str.split('|').explode()
genre_counts = genre_series.value_counts().head(15)

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(genre_counts.index, genre_counts.values, color='steelblue')
ax.invert_yaxis()
ax.set_xlabel('Number of movies')
ax.set_title('Top 15 genres in the catalog')
plt.tight_layout()
plt.savefig('results/figures/genre_distribution.png', dpi=150)
plt.show()

Drama and Comedy dominate the catalog. This is relevant for the content-based recommender: if we use raw genre counts without TF-IDF, Drama would be overrepresented and would pull most user profiles towards Drama movies regardless of actual preference. TF-IDF down-weights common genres exactly to address this.

## 7. Train / test split

We split ratings 80/20 at random. A more rigorous approach would use a temporal split (train on earlier ratings, test on later ones), but for this prototype a random split is sufficient.

In [ ]:
import os
os.makedirs('results/figures', exist_ok=True)

train, test = train_test_split_ratings(ratings, test_size=0.2)
print(f"Train : {len(train):,} ratings")
print(f"Test  : {len(test):,} ratings")
print(f"Split : {len(train)/len(ratings):.0%} / {len(test)/len(ratings):.0%}")

The train and test sets are used consistently across all models in the following notebooks, so comparisons between methods are fair.